# Module 07 — Walk-Forward Out-of-Sample Backtest

This notebook runs the final **30% out-of-sample** strategy simulation.

Frozen from the 70% formation sample:

\[
\alpha,\beta,H,\mu,\sigma,\kappa,\operatorname{Var}(X)
\]

and the final 40 structurally eligible pairs.

Updated through each OOS date using information available at that date:

- current log-price spread and Z-score,
- conditional fOU convergence horizon,
- EWMA volatility,
- risk-free rate,
- synthetic option values.

Entry requires:

\[
|Z_t|\ge 1.5
\]

and a conditional convergence horizon satisfying:

\[
T_t^{70}\le126\text{ trading days}.
\]

Only one open trade per pair is allowed.


In [ ]:

import numpy as np
import pandas as pd


# Existing source/module locations.
candidate_src = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "corrected_fou_pipeline",
    PROJECT_ROOT / "fou_modules",
    Path("/mnt/data/corrected_fou_pipeline"),
    Path("/mnt/data/fou_modules"),
    Path("/mnt/data/module_07_backtest"),
]
for p in candidate_src:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from backtest import run_walk_forward_backtest, backtest_summary

pd.set_option("display.max_columns", 100)


## 1. Load frozen formation data

The date index of both price files **must** be a genuine `DatetimeIndex`.
Do not continue if the parquet files were saved with `index=False`.


In [ ]:
DATA = PROJECT_ROOT / "data" / "processed"

train_prices = pd.read_parquet(DATA / "train_prices.parquet")
test_prices = pd.read_parquet(DATA / "test_prices.parquet")
eligible_pairs = pd.read_parquet(DATA / "eligible_pairs.parquet")
cointegrated_pairs = pd.read_parquet(DATA / "cointegrated_pairs.parquet")

assert isinstance(train_prices.index, pd.DatetimeIndex), "train_prices index is not DatetimeIndex."
assert isinstance(test_prices.index, pd.DatetimeIndex), "test_prices index is not DatetimeIndex."
assert train_prices.index.max() < test_prices.index.min(), "Train/test periods overlap."

print("Train:", train_prices.index.min(), "->", train_prices.index.max(), train_prices.shape)
print("Test :", test_prices.index.min(), "->", test_prices.index.max(), test_prices.shape)
print("Eligible pairs:", len(eligible_pairs))


## 2. Historical risk-free rate

Module 07 expects the rate in **decimal form**, e.g. `0.0425` for 4.25%.

If Module 06 already saved the historical rate series, point `RF_PATH` to that file.
The backtest always uses the latest rate observable **on or before** each OOS date.

A scalar rate is also accepted technically, but the final thesis backtest should use
the historical series prepared in Module 06.


In [ ]:
# Change this path only if Module 06 saved it under a different filename.
RF_PATH = DATA / "risk_free_rates.parquet"

if not RF_PATH.exists():
    raise FileNotFoundError(
        f"{RF_PATH} not found. Save the historical risk-free series from Module 06 "
        "as a dated parquet before running the final backtest."
    )

rf_data = pd.read_parquet(RF_PATH)

# Accept either a one-column DataFrame or a Series saved to parquet.
if isinstance(rf_data, pd.DataFrame):
    if rf_data.shape[1] != 1:
        raise ValueError("risk_free_rates.parquet must contain exactly one rate column.")
    risk_free_rates = rf_data.iloc[:, 0]
else:
    risk_free_rates = pd.Series(rf_data)

risk_free_rates.index = pd.to_datetime(risk_free_rates.index)
risk_free_rates = risk_free_rates.sort_index().astype(float)

# Safety check: values should already be decimals, not percentages.
if risk_free_rates.abs().median() > 1:
    raise ValueError("Risk-free rates appear to be percentages. Divide them by 100 first.")

risk_free_rates.tail()


## 3. Final backtest configuration

`INITIAL_CAPITAL = $100,000` is a portfolio funding assumption, not a parameter
optimized to improve results. Long options are paid fully from cash; the strategy
does not borrow or use leverage.

If the minimum beta/delta hedge unit cannot be funded from available cash, that
signal is recorded as `insufficient_cash` and skipped.


In [ ]:
INITIAL_CAPITAL = 100_000.0

ENTRY_Z = 1.5
TARGET_PROBABILITY = 0.70
MEMORY_WINDOW = 60
MAX_HORIZON_DAYS = 126
N_PATHS = 5000
EWMA_LAMBDA = 0.94
SEED = 42


## 4. Run the walk-forward simulation

The expensive conditional fOU Monte Carlo is only called when:

1. the pair has no open position, and
2. the current absolute Z-score is at least 1.5.

This avoids simulating convergence probabilities for economically irrelevant
near-equilibrium observations.


In [ ]:
results = run_walk_forward_backtest(
    train_prices=train_prices,
    test_prices=test_prices,
    eligible_pairs=eligible_pairs,
    cointegrated_pairs=cointegrated_pairs,
    risk_free_rates=risk_free_rates,
    initial_capital=INITIAL_CAPITAL,
    entry_z=ENTRY_Z,
    target_probability=TARGET_PROBABILITY,
    memory_window=MEMORY_WINDOW,
    max_horizon_days=MAX_HORIZON_DAYS,
    n_paths=N_PATHS,
    ewma_lambda=EWMA_LAMBDA,
    seed=SEED,
)

trades = results["trades"]
equity_curve = results["equity_curve"]
skipped_signals = results["skipped_signals"]

print("Completed trades:", len(trades))
print("Skipped records :", len(skipped_signals))
print("Final equity    :", equity_curve["equity"].iloc[-1])


## 5. Sanity checks

These are implementation checks, not the final Results section.


In [ ]:
summary = backtest_summary(
    trades=trades,
    equity_curve=equity_curve,
    initial_capital=INITIAL_CAPITAL,
)
summary


In [ ]:
if not trades.empty:
    display(
        trades[
            [
                "pair",
                "entry_date",
                "exit_date",
                "entry_z",
                "convergence_horizon_trading_days",
                "option_calendar_dte",
                "dependent_contracts",
                "independent_contracts",
                "entry_premium",
                "exit_value",
                "pnl",
                "trade_return",
                "exit_reason",
            ]
        ].head(20)
    )


In [ ]:
if not skipped_signals.empty and "reason" in skipped_signals.columns:
    display(skipped_signals["reason"].value_counts())


In [ ]:
# Capital adequacy diagnostic.
if not trades.empty:
    print("Largest single entry premium:", trades["entry_premium"].max())
    print("Median entry premium:", trades["entry_premium"].median())
print("Maximum concurrent positions:", equity_curve["n_open_positions"].max())
print("Minimum cash balance:", equity_curve["cash"].min())


## 6. Save Module 07 outputs

`trades.parquet` is the main input to Module 08 (Results).

`equity_curve.parquet` contains the daily cash, open-position value, total equity,
and number of concurrent positions.

`skipped_signals.parquet` makes the capital constraint auditable.


In [ ]:
trades.to_parquet(DATA / "trades.parquet", index=False)
equity_curve.to_parquet(DATA / "equity_curve.parquet", index=True)
skipped_signals.to_parquet(DATA / "skipped_signals.parquet", index=False)

print("Saved:")
print(DATA / "trades.parquet")
print(DATA / "equity_curve.parquet")
print(DATA / "skipped_signals.parquet")
